# NOTEBOOK: 02 — Silver Standardization

- **Layer:** Silver
- **Purpose:** Read Bronze raw tables and apply cleaning and normalization to produce trusted Silver tables.
- **Inputs:** 
    - `main.techmart_bronze.raw_products`
    - `main.techmart_bronze.raw_vendors`
- **Outputs:** 
    - `main.techmart_silver.products`
    - `main.techmart_silver.vendors`
- **Transforms:** 
    - Products
        - weight to kg
        - price to USD, 
        - description cleaned. 
        - duplicates dropped.
    - Vendors
        - vendor name: legal suffixes removed, canonical mapping applied
        - duplicates dropped.
- **Author:** Maira Tavares

# 0. Imports

In [0]:
import re
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StringType

In [0]:
from utils.config import (
    BRONZE_PRODUCTS,
    BRONZE_VENDORS,
    SILVER_PRODUCTS,
    SILVER_VENDORS
)

PROCESSED_AT = datetime.now().isoformat()

print("✅ Config loaded")
print(f"  Source  : {BRONZE_PRODUCTS}, {BRONZE_VENDORS}")
print(f"  Target  : {SILVER_PRODUCTS}, {SILVER_VENDORS}")
print(f"  Processed at: {PROCESSED_AT}")

# 1. Functions

In [0]:
# Transformation functions
# These functions are defined here because Spark UDF serialization
# requires functions to be in the notebook scope.
# All functions are independently tested in tests/test_transformations.py


In [0]:
def parse_weight_to_kg(raw):
    """
    Parses any weight string and returns a float in kg.
    Handles 15+ formats: kg, KG, Kg., grams, gr, grs, G., Kilograms etc.
    Returns None if the value cannot be parsed.
    """
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None

    raw = str(raw).strip().lower().rstrip(".")

    # Extract numeric part — removes all non-numeric characters except . and ,
    numeric_str = re.sub(r"[^\d.,]", " ", raw).strip()
    numeric_str = numeric_str.replace(",", ".").replace(" ", "")

    try:
        value = float(numeric_str)
    except:
        return None

    # Detect unit and convert to kg
    if any(unit in raw for unit in ["kilogram", "kilograms"]):
        return round(value, 4)
    elif "kg" in raw:
        return round(value, 4)
    elif any(unit in raw for unit in ["gram", "grams", "gr", "grs", "g.", " g"]):
        return round(value / 1000, 4)
    elif raw[-1] == "g":
        return round(value / 1000, 4)
    else:
        return round(value, 4)

parse_weight_udf      = udf(parse_weight_to_kg, DoubleType())


In [0]:
def parse_price_to_usd(raw):
    """
    Parses any price string and returns a float in USD.
    Handles 10+ formats: $, USD, dollars, comma decimals, space decimals etc.
    Returns None if the value cannot be parsed.
    """
    if raw is None or str(raw).strip().lower() in ("none", "nan", ""):
        return None

    raw = str(raw).strip().lower()
    raw = raw.replace("$", "").replace("usd", "").replace("dollars", "").strip()

    # Comma + period → comma is thousands separator → "1,099.00" → "1099.00"
    if "," in raw and "." in raw:
        raw = raw.replace(",", "")
    # Comma only + 2 digits after → decimal separator → "349,00" → "349.00"
    elif "," in raw and "." not in raw:
        parts = raw.split(",")
        if len(parts) == 2 and len(parts[1].strip()) <= 2:
            raw = raw.replace(",", ".")
        else:
            raw = raw.replace(",", "")

    # Space as decimal separator → "59 99" → "59.99"
    space_decimal = re.match(r"^(\d+)\s(\d{2})$", raw.strip())
    if space_decimal:
        raw = f"{space_decimal.group(1)}.{space_decimal.group(2)}"
    else:
        raw = raw.replace(" ", "")

    try:
        return round(float(raw), 2)
    except:
        return None

parse_price_udf       = udf(parse_price_to_usd, DoubleType())    

In [0]:
def normalize_product_description(raw):
    """
    Normalizes product description to clean lowercase text.
    Collapses extra spaces and strips leading/trailing whitespace.
    Output is used as input for LLM extraction in Stage 3.
    """
    if raw is None:
        return None
    cleaned = str(raw).strip().lower()
    if cleaned in ("", "none", "nan", "null"):
        return None
    return " ".join(cleaned.split())


normalize_desc_udf    = udf(normalize_product_description, StringType())

In [0]:
def remove_suffixes(text: str) -> str:
    """
    Removes legal entity suffixes only at the END of the string.
    Uses regex $ anchor to prevent removing patterns from inside words.
    Example: "sony corporation" → "sony" (not "sony rporation")
    """
    suffixes = [
        r"\s+corporation$", r"\s+corp\.$", r"\s+corp$",
        r"\s+incorporated$", r"\s+inc\.$", r"\s+inc$",
        r"\s+limited$", r"\s+ltd\.$", r"\s+ltd$",
        r"\s+co\.$", r"\s+co$", r",$"
    ]
    for suffix in suffixes:
        text = re.sub(suffix, "", text, flags=re.IGNORECASE).strip()
    return text

In [0]:
def normalize_vendor(raw):
    """
    Normalizes vendor names in two steps:
    Step 1 — Generic cleaning: lowercase, collapse spaces, remove legal suffixes
    Step 2 — Canonical mapping: fix known abbreviations and brand variations
    """
    if raw is None:
        return None
    cleaned = str(raw).strip()
    if cleaned == "" or cleaned.lower() in ("none", "nan", "null"):
        return None

    # Step 1 — Generic cleaning
    normalized = " ".join(cleaned.split()).lower()
    normalized = remove_suffixes(normalized)
    normalized = normalized.rstrip(".").strip()
    normalized = normalized.title()

    # Step 2 — Canonical mapping for known brand variations
    canonical_map = {
        "Amazon.Com"                : "Amazon",
        "Asus Tek"                  : "Asus",
        "Asus Tek Computer"         : "Asus",
        "Hyperx (Kingston)"         : "HyperX (Kingston)",
        "Hp"                        : "HP",
        "Dji Technology"            : "DJI Technology",
        "Logitech International S.A": "Logitech",
    }
    return canonical_map.get(normalized, normalized)

normalize_vendor_udf  = udf(normalize_vendor, StringType())

# 2. Clean Product

In [0]:
# Clean and normalize products
# Applies all transformation UDFs and selects final Silver columns.
# product_description_raw is preserved for lineage — Silver keeps both
# the original and the cleaned version.

df_products_silver = (
    spark.table(BRONZE_PRODUCTS)
    .withColumn("weight_kg",           parse_weight_udf(F.col("weight_raw")))
    .withColumn("unit_price_usd",      parse_price_udf(F.col("unit_price_raw")))
    .withColumn("product_description", normalize_desc_udf(F.col("product_description_raw")))
    .dropDuplicates(["product_id"])
    .select(
        F.col("product_id").cast("integer"),
        F.col("product_description_raw"),
        F.col("product_description"),
        F.col("weight_kg"),
        F.col("unit_price_usd"),
        F.col("_ingested_at"),
        F.col("_source_system"),
        F.lit(PROCESSED_AT).alias("_processed_at")
    )
)

print("✅ Products cleaned")
display(df_products_silver)

In [0]:
# Write Silver Products table
(df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PRODUCTS))

# Validate nulls — nulls in weight/price mean the parser failed for those rows
null_weights = spark.table(SILVER_PRODUCTS).filter(F.col("weight_kg").isNull()).count()
null_prices  = spark.table(SILVER_PRODUCTS).filter(F.col("unit_price_usd").isNull()).count()

print(f"✅ Written : {SILVER_PRODUCTS}")
print(f"   Rows    : {spark.table(SILVER_PRODUCTS).count()}")
print(f"   Null weights : {null_weights}")
print(f"   Null prices  : {null_prices}")

# 3. Clean Vendors

In [0]:
# Clean and normalize vendors
# Deduplicates on product_id + vendor_name to handle cases where
# the same vendor appears multiple times for the same product.
# vendor_name_raw is preserved for auditability.

df_vendors_silver = (
    spark.table(BRONZE_VENDORS)
    .withColumn("vendor_name", normalize_vendor_udf(F.col("vendor_name_raw")))
    .dropDuplicates(["product_id", "vendor_name"])
    .select(
        F.col("product_id").cast("integer"),
        F.col("vendor_name_raw"),
        F.col("vendor_name"),
        F.col("_ingested_at"),
        F.lit(PROCESSED_AT).alias("_processed_at")
    )
)

print("✅ Vendors cleaned")
display(df_vendors_silver.select("product_id", "vendor_name_raw", "vendor_name"))

In [0]:
# Write Silver Vendors table
(df_vendors_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_VENDORS))

print(f"✅ Written : {SILVER_VENDORS}")
print(f"   Rows    : {spark.table(SILVER_VENDORS).count()}")